# AgriCare AI — Notebook 06: Two-Stage Model Training & Fine-Tuning

**Project:** AgriCare AI (LeafAgro)  
**Phase:** Phase 38 — AI Model Training & Fine-Tuning  

---
### Overview
This interactive notebook demonstrates the complete two-stage transfer learning pipeline for AgriCare AI:
1. **Pre-Training Readiness & Leakage Audit**
2. **Streaming Data Loaders (Train & Validation only; Test split isolated)**
3. **Stage A: Feature Extraction (Frozen MobileNetV2 backbone)**
4. **Stage B: Controlled Fine-Tuning (Upper backbone unfreezing, BatchNorm frozen)**
5. **Validation-Based Model Comparison & Selection**
6. **Inspection of Training Curves and Convergence Artifacts**

## 1. Environment & Module Imports
We import TensorFlow, our project configuration, and the training engine.

In [ ]:
import sys
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

# Add ai_model directory to sys.path
nb_dir = Path.cwd()
ai_model_dir = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir / "ai_model"
if str(ai_model_dir) not in sys.path:
    sys.path.insert(0, str(ai_model_dir))

from src.config import (
    PROJECT_ROOT, BEST_MODEL_PATH, STAGE_A_CHECKPOINT_PATH,
    STAGE_B_CHECKPOINT_PATH, TRAINING_HISTORY_PATH, LABELS_PATH,
    MODEL_METADATA_PATH, NUM_CLASSES
)
from src.training_utils import run_pre_training_audit, analyze_class_distribution
from src.dataset_loader import load_train_dataset, load_validation_dataset

print(f"TensorFlow: {tf.__version__}")
print(f"Best Model Checkpoint: {BEST_MODEL_PATH.name}")

## 2. Pre-Training Readiness & Leakage Audit
We verify that zero data leakage exists between train and validation splits.

In [ ]:
audit = run_pre_training_audit()
print(f"Pre-Training Audit Status: {audit['status']}")
print(f"Train Samples: {audit['dataset_counts']['train_samples']}")
print(f"Validation Samples: {audit['dataset_counts']['validation_samples']}")
print(f"Test Samples (Isolated): {audit['dataset_counts']['test_samples']}")

## 3. Class Distribution & Imbalance Analysis
Let us inspect the distribution of samples across active crop categories.

In [ ]:
dist = analyze_class_distribution()
print(f"Active Classes in Pilot: {dist['active_classes_count']}")
print(f"Imbalance Ratio: {dist['imbalance_ratio']}:1")
for cls_name, count in sorted(dist['train_counts'].items()):
    print(f"  {cls_name:<32}: {count} train, {dist['validation_counts'].get(cls_name, 0)} val")

## 4. Inspecting Consolidated Training History
Let us load `training_history.csv` to review the loss and accuracy metrics achieved across Stage A and Stage B.

In [ ]:
if TRAINING_HISTORY_PATH.exists():
    df = pd.read_csv(TRAINING_HISTORY_PATH)
    print(f"Total epochs recorded: {len(df)}")
    display(df.tail(10))
else:
    print("Run ai_model/src/train.py first to generate training history.")

## 5. Visualizing Accuracy & Loss Trajectories
We plot the convergence curves comparing training and validation performance.

In [ ]:
if TRAINING_HISTORY_PATH.exists():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    ax1.plot(df['global_epoch'], df['accuracy'], 'o-', label='Train Accuracy', color='#2E7D32')
    if 'val_accuracy' in df.columns:
        ax1.plot(df['global_epoch'], df['val_accuracy'], 's--', label='Val Accuracy', color='#1565C0')
    ax1.set_title('Training vs. Validation Accuracy')
    ax1.set_xlabel('Global Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend()
    
    # Loss
    ax2.plot(df['global_epoch'], df['loss'], 'o-', label='Train Loss', color='#C62828')
    if 'val_loss' in df.columns:
        ax2.plot(df['global_epoch'], df['val_loss'], 's--', label='Val Loss', color='#EF6C00')
    ax2.set_title('Training vs. Validation Loss')
    ax2.set_xlabel('Global Epoch')
    ax2.set_ylabel('Cross-Entropy Loss')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Verifying Deployed Best Model & Sample Inference
We load `models/best_model.keras` and run a real validation batch through the network to verify output dimensions and valid softmax probabilities.

In [ ]:
best_model = tf.keras.models.load_model(str(BEST_MODEL_PATH))
print(f"Loaded: {BEST_MODEL_PATH.name}")
print(f"Input: {best_model.input_shape}, Output: {best_model.output_shape}")

with open(LABELS_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)
    class_names = labels_dict['class_names']

val_ds = load_validation_dataset(batch_size=4)
for images, true_labels in val_ds.take(1):
    preds = best_model(images, training=False).numpy()
    top_preds = preds.argmax(axis=-1)
    
    for i in range(len(images)):
        true_cls = class_names[true_labels[i].numpy()]
        pred_cls = class_names[top_preds[i]]
        conf = preds[i][top_preds[i]] * 100
        print(f"Sample {i+1} — True: {true_cls} | Predicted: {pred_cls} ({conf:.1f}% confidence)")